# Notebook 21 — Causal Forest CATE Estimation
### Heterogeneous Treatment Effects in Mortgage Lending

**Author:** Rajveer Singh Pall

Trains CausalForestDML to estimate individual CATE for every applicant.

**INPUT:** `data/features_panel.parquet`, `data/trim_bounds.json`, `data/feature_sets.json`

**OUTPUTS:** `data/cate_estimates.parquet`, `outputs/figures/nb21_cate_distribution.png`, `outputs/tables/nb21_cate_summary.csv`

**RUNTIME:** 60-90 minutes on i7-13650HX

In [ ]:
# CELL 1 - IMPORTS
import pandas as pd
import numpy as np
import polars as pl
import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json, gc, time, warnings
from pathlib import Path
from econml.dml import CausalForestDML

warnings.filterwarnings('ignore')

BASE_DIR    = Path('D:/CATE-HMDA-Heterogeneous-Effects')
DATA_DIR    = BASE_DIR / 'data'
TABLES_DIR  = BASE_DIR / 'outputs' / 'tables'
FIGURES_DIR = BASE_DIR / 'outputs' / 'figures'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 150, 'font.family': 'serif', 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})

with open(DATA_DIR / 'feature_sets.json') as f:
    feature_sets = json.load(f)
X_FULL   = feature_sets['X_FULL']
X_HETERO = feature_sets['X_HETERO']

with open(DATA_DIR / 'trim_bounds.json') as f:
    trim_bounds = json.load(f)

import econml
print(f'econml {econml.__version__} loaded')
print(f'X_FULL: {len(X_FULL)} features')
print(f'Trim bounds from NB18: [{trim_bounds["trim_lo"]:.4f}, {trim_bounds["trim_hi"]:.4f}]')

In [ ]:
# CELL 2 - LOAD SAMPLE (1.5M rows, different seed from NB19)
print('='*70)
print('LOADING SAMPLE')
print('='*70)

N_BLACK = 300_000
N_WHITE = 1_200_000

lf = pl.scan_parquet(str(DATA_DIR / 'features_panel.parquet'))
df_black = lf.filter(pl.col('black') == 1).collect().sample(n=N_BLACK, seed=123)
df_white = lf.filter(pl.col('black') == 0).collect().sample(n=N_WHITE, seed=123)
df = pl.concat([df_black, df_white]).to_pandas()
del df_black, df_white, lf
gc.collect()

df = df.sample(frac=1, random_state=123).reset_index(drop=True)
X_use = [f for f in X_FULL if f in df.columns]

X = df[X_use].values.astype(np.float32)
T = df['black'].values.astype(np.float32)
Y = df['approved'].values.astype(np.float32)

raw_gap = 100 * (df[df['black']==0]['approved'].mean() - df[df['black']==1]['approved'].mean())
print(f'Sample  : {len(df):,} rows')
print(f'Black   : {T.sum():.0f} ({100*T.mean():.1f}%)')
print(f'Raw gap : {raw_gap:.2f} pp')
print(f'RAM     : {df.memory_usage(deep=True).sum()/1e6:.0f} MB')

In [ ]:
# CELL 3 - DEFINE NUISANCE MODELS
model_t = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.05, num_leaves=31,
    min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=42, verbose=-1
)
model_y = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.05, num_leaves=31,
    min_child_samples=50, subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=42, verbose=-1
)
print('Nuisance models defined')
print('  model_t: LightGBM (predicts black from X)')
print('  model_y: LightGBM (predicts approved from X)')

In [ ]:
# CELL 4 - TRAIN CAUSAL FOREST
# RUNTIME: 45-75 minutes. The notebook will appear frozen during training.
# Do NOT interrupt. You will see output only after training completes.
print('='*70)
print('TRAINING CAUSAL FOREST')
print('500 trees, 5-fold cross-fitting, honest splitting')
print('Expected: 45-75 minutes. Do not interrupt.')
print('='*70)

t0 = time.time()

forest = CausalForestDML(
    model_t=model_t,
    model_y=model_y,
    n_estimators=500,
    n_crossfit_splits=5,
    honest=True,
    inference=True,
    discrete_treatment=True,
    discrete_outcome=False,
    n_jobs=-1,
    random_state=42,
    verbose=0,
)

forest.fit(Y, T, X=X)

elapsed = time.time() - t0
print(f'Trained in {elapsed/60:.1f} minutes')

ate_forest = forest.ate(X) * 100
print(f'Forest ATE : {ate_forest:.2f} pp  (NB19 DML: -9.39 pp)')
print(f'Difference : {abs(ate_forest - (-9.39)):.2f} pp')
print('Sanity check: ✅' if abs(ate_forest - (-9.39)) < 5 else 'Sanity check: INVESTIGATE')

In [ ]:
# CELL 5 - COMPUTE INDIVIDUAL CATE ESTIMATES
print('='*70)
print('COMPUTING INDIVIDUAL CATE ESTIMATES')
print('='*70)

t0 = time.time()
cate_raw = forest.effect(X)
cate_pp  = cate_raw * 100

print('Computing confidence intervals...')
ci       = forest.effect_interval(X, alpha=0.05)
cate_lo  = ci[0].flatten() * 100
cate_hi  = ci[1].flatten() * 100
print(f'Done in {time.time()-t0:.0f}s')

df['cate_pp'] = cate_pp
df['cate_lo'] = cate_lo
df['cate_hi'] = cate_hi

pct_negative = 100 * (cate_pp < 0).mean()

print(f'CATE Distribution:')
print(f'  Mean   : {cate_pp.mean():+.2f} pp')
print(f'  Std    : {cate_pp.std():.2f} pp  <- HETEROGENEITY')
print(f'  Min    : {cate_pp.min():+.2f} pp')
print(f'  Max    : {cate_pp.max():+.2f} pp')
print(f'  P10    : {np.percentile(cate_pp, 10):+.2f} pp')
print(f'  P25    : {np.percentile(cate_pp, 25):+.2f} pp')
print(f'  P75    : {np.percentile(cate_pp, 75):+.2f} pp')
print(f'  P90    : {np.percentile(cate_pp, 90):+.2f} pp')
print(f'  % penalised: {pct_negative:.1f}%')

In [ ]:
# CELL 6 - CATE DISTRIBUTION FIGURE (KEY FIGURE OF THE PAPER)
print('Generating distribution figure...')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.hist(cate_pp, bins=100, color='#1565C0', alpha=0.8, edgecolor='none')
ax.axvline(cate_pp.mean(), color='#E53935', linewidth=2,
           label=f'Mean = {cate_pp.mean():.2f} pp')
ax.axvline(0, color='black', linewidth=1, linestyle='--', label='Zero')
ax.axvline(np.percentile(cate_pp, 10), color='#FF6F00', linewidth=1.5,
           linestyle=':', label=f'P10 = {np.percentile(cate_pp,10):.2f} pp')
ax.axvline(np.percentile(cate_pp, 90), color='#2E7D32', linewidth=1.5,
           linestyle=':', label=f'P90 = {np.percentile(cate_pp,90):.2f} pp')
ax.set_xlabel('Estimated approval penalty (pp)')
ax.set_ylabel('Number of applicants')
ax.set_title('CATE distribution: individual approval penalties')
ax.legend(fontsize=9)
ax.text(0.02, 0.85,
        f'n={len(cate_pp):,}\nStd={cate_pp.std():.2f} pp\n{pct_negative:.0f}% penalised',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

ax = axes[1]
df['income_q'] = pd.qcut(df['income'], q=5,
                          labels=['Q1 (lowest)', 'Q2', 'Q3', 'Q4', 'Q5 (highest)'])
cate_by_income = df.groupby('income_q', observed=True)['cate_pp'].agg(['mean','std','count'])
cate_by_income['se'] = cate_by_income['std'] / np.sqrt(cate_by_income['count'])

ax.bar(range(5), cate_by_income['mean'], color='#1565C0', alpha=0.8)
ax.errorbar(range(5), cate_by_income['mean'],
            yerr=1.96*cate_by_income['se'],
            fmt='none', color='black', capsize=4, linewidth=1.5)
ax.axhline(cate_pp.mean(), color='#E53935', linestyle='--', linewidth=1.5,
           label=f'Overall mean ({cate_pp.mean():.2f} pp)')
ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
ax.set_xticks(range(5))
ax.set_xticklabels(cate_by_income.index, fontsize=9)
ax.set_xlabel('Income quintile')
ax.set_ylabel('Mean CATE (pp)')
ax.set_title('Mean CATE by income quintile')
ax.legend(fontsize=9)

plt.suptitle('Figure NB21: Causal Forest CATE Distribution', fontsize=12)
plt.tight_layout()
out = FIGURES_DIR / 'nb21_cate_distribution.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out.name}')

In [ ]:
# CELL 7 - CATE BY SUBGROUP
print('='*70)
print('CATE BY SUBGROUP')
print('='*70)

subgroup_results = []

def sg_stats(mask, label):
    cate_sub = cate_pp[mask]
    n = mask.sum()
    mean = cate_sub.mean()
    se = cate_sub.std() / np.sqrt(n)
    return {'subgroup': label, 'n': n, 'mean_cate': round(mean, 3),
            'se': round(se, 4), 'ci_lo': round(mean-1.96*se, 3),
            'ci_hi': round(mean+1.96*se, 3),
            'pct_penalised': round(100*(cate_sub < 0).mean(), 1)}

# Income quintiles
income_q = pd.qcut(df['income'], q=5, labels=False)
for q in range(5):
    subgroup_results.append(sg_stats((income_q == q).values, f'Income Q{q+1}'))

# LTV
subgroup_results.append(sg_stats((df['ltv'] <= 80).values, 'LTV <= 80%'))
subgroup_results.append(sg_stats((df['ltv'] > 80).values, 'LTV > 80%'))

# Loan purpose
if 'purpose_purchase' in df.columns:
    subgroup_results.append(sg_stats(df['purpose_purchase'].values == 1, 'Purchase loans'))
    subgroup_results.append(sg_stats(df['purpose_refi'].values == 1, 'Refinance loans'))

# Lender size
subgroup_results.append(sg_stats(df['lender_small'].values == 1, 'Small lenders'))
subgroup_results.append(sg_stats(df['lender_large'].values == 1, 'Large lenders'))

# Pre/post tightening
subgroup_results.append(sg_stats(df['post_tightening'].values == 0, 'Pre-2022'))
subgroup_results.append(sg_stats(df['post_tightening'].values == 1, 'Post-2022'))

# AUS
if 'aus_automated' in df.columns:
    subgroup_results.append(sg_stats(df['aus_automated'].values == 1, 'Automated AUS'))
    subgroup_results.append(sg_stats(df['aus_exempt'].values == 1, 'Manual/exempt AUS'))

# DTI
if 'dti_high' in df.columns:
    subgroup_results.append(sg_stats(df['dti_high'].values == 0, 'Low DTI (<43%)'))
    subgroup_results.append(sg_stats(df['dti_high'].values == 1, 'High DTI (>=43%)'))

sg_df = pd.DataFrame(subgroup_results)
print(sg_df[['subgroup','n','mean_cate','se','pct_penalised']].sort_values('mean_cate').to_string(index=False))

In [ ]:
# CELL 8 - SUBGROUP FIGURE
print('Generating subgroup figure...')

plot_groups = ['Income Q1', 'Income Q3', 'Income Q5',
               'LTV <= 80%', 'LTV > 80%',
               'Purchase loans', 'Refinance loans',
               'Automated AUS', 'Manual/exempt AUS',
               'Pre-2022', 'Post-2022']
plot_df = sg_df[sg_df['subgroup'].isin(plot_groups)].sort_values('mean_cate').copy()

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#E53935' if v < cate_pp.mean() else '#1565C0' for v in plot_df['mean_cate']]
ax.barh(range(len(plot_df)), plot_df['mean_cate'], color=colors, alpha=0.8)
ax.errorbar(plot_df['mean_cate'], range(len(plot_df)),
            xerr=1.96*plot_df['se'], fmt='none', color='black', capsize=3, linewidth=1.2)
ax.axvline(cate_pp.mean(), color='black', linewidth=1.5, linestyle='--',
           label=f'Overall mean ({cate_pp.mean():.2f} pp)')
ax.axvline(0, color='gray', linewidth=0.8, linestyle=':')
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df['subgroup'], fontsize=9)
ax.set_xlabel('Mean CATE (pp) with 95% CI')
ax.set_title('CATE by Subgroup\nRed = larger penalty than average')
ax.legend(fontsize=9)
for i, (_, row) in enumerate(plot_df.iterrows()):
    ax.text(row['mean_cate'] - 0.2, i, f"{row['mean_cate']:.2f}",
            va='center', ha='right', fontsize=8)
plt.tight_layout()
out = FIGURES_DIR / 'nb21_cate_by_subgroup.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out.name}')

In [ ]:
# CELL 9 - SAVE OUTPUTS
print('='*70)
print('SAVING OUTPUTS')
print('='*70)

save_cols = ['year', 'black', 'approved', 'income', 'ltv', 'cate_pp', 'cate_lo', 'cate_hi']
extra = ['dti_midpoint', 'purpose_purchase', 'purpose_refi',
         'aus_automated', 'aus_exempt', 'lender_small', 'lender_large', 'post_tightening']
save_cols += [c for c in extra if c in df.columns]

pl.from_pandas(df[save_cols]).write_parquet(str(DATA_DIR / 'cate_estimates.parquet'))
print(f'Saved: cate_estimates.parquet ({len(df):,} rows)')

sg_df.to_csv(TABLES_DIR / 'nb21_cate_subgroups.csv', index=False)
print('Saved: nb21_cate_subgroups.csv')

summary = pd.DataFrame([{
    'n_sample': len(df),
    'forest_ate_pp': round(float(ate_forest), 4),
    'cate_mean_pp': round(float(cate_pp.mean()), 4),
    'cate_median_pp': round(float(np.median(cate_pp)), 4),
    'cate_std_pp': round(float(cate_pp.std()), 4),
    'cate_p10_pp': round(float(np.percentile(cate_pp, 10)), 4),
    'cate_p90_pp': round(float(np.percentile(cate_pp, 90)), 4),
    'pct_penalised': round(float(pct_negative), 1),
}])
summary.to_csv(TABLES_DIR / 'nb21_cate_summary.csv', index=False)
print('Saved: nb21_cate_summary.csv')
print(summary.T.to_string())

In [ ]:
# CELL 10 - VERIFICATION CHECKS
print('='*70)
print('VERIFICATION CHECKS')
print('='*70)

diff_nb19 = abs(float(ate_forest) - (-0.0939))
print(f'1. Forest ATE: {ate_forest:.4f} ({ate_forest*100:.2f} pp)')
assert diff_nb19 < 0.05, f'Forest ATE too far from NB19: {diff_nb19}'
print(f'   Consistent with NB19 (diff={diff_nb19:.4f})')

assert cate_pp.std() > 1.0
print(f'2. CATE std = {cate_pp.std():.2f} pp -> meaningful heterogeneity')

assert pct_negative > 50
print(f'3. {pct_negative:.1f}% penalised -> majority face negative CATE')

for f in ['cate_estimates.parquet']:
    assert (DATA_DIR / f).exists()
for f in ['nb21_cate_summary.csv', 'nb21_cate_subgroups.csv']:
    assert (TABLES_DIR / f).exists()
print(f'4. All output files present')

income_q1 = sg_df[sg_df['subgroup']=='Income Q1']['mean_cate'].values[0]
income_q5 = sg_df[sg_df['subgroup']=='Income Q5']['mean_cate'].values[0]
print(f'\n5. KEY FINDING:')
print(f'   Low-income (Q1) CATE  : {income_q1:.2f} pp')
print(f'   High-income (Q5) CATE : {income_q5:.2f} pp')
if abs(income_q5) > abs(income_q1):
    print(f'   Gap LARGER for high-income -> consistent with animus')
else:
    print(f'   Gap LARGER for low-income -> consistent with structural bias')

print('\n' + '='*70)
print('ALL VERIFICATION CHECKS PASSED')
print(f'Forest ATE : {ate_forest*100:.2f} pp')
print(f'CATE mean  : {cate_pp.mean():.2f} pp')
print(f'CATE std   : {cate_pp.std():.2f} pp')
print(f'P10 / P90  : {np.percentile(cate_pp,10):.2f} / {np.percentile(cate_pp,90):.2f} pp')
print(f'% penalised: {pct_negative:.1f}%')
print('NB21 complete -> proceed to NB22_shap_attribution.ipynb')
print('='*70)